In [ ]:
import sys
!{sys.executable} -m pip install -q DECIMER rdkit pubchempy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 24.4 MB/s eta 0:00:00


In [ ]:
import os, cv2, numpy as np, pandas as pd
from PIL import Image
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

IMAGE_DIR   = "/content"                 # folder with your images
OUT_DIR     = "/content/output"          # results go here
TRY_HAND_DRAWN = True                    # also try DECIMER's hand-drawn model (slower, helps sketches)
USE_PUBCHEM    = True                    # verify reads against PubChem (needs internet)

# OPTIONAL: known correct SMILES to measure real accuracy. Leave {} if you don't have them.
GROUND_TRUTH = {
    # "molecule": "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",
    # "mix1_aspirin_lowres": "CC(=O)Oc1ccccc1C(=O)O",
}
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def _strip_frame(gray, dark=70, area_frac=0.03):
    mask=(gray<dark).astype(np.uint8); n,lab,st,_=cv2.connectedComponentsWithStats(mask,8)
    H,W=gray.shape; out=gray.copy()
    for i in range(1,n):
        x,y,w,h,a=st[i]
        if (x==0 or y==0 or x+w==W or y+h==H) and a>area_frac*H*W: out[lab==i]=255
    return out

def clean_image(src, binarize=True):
    img=cv2.imread(src) if isinstance(src,str) else src
    if img is None: raise FileNotFoundError(src)
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) if img.ndim==3 else img
    H,W=gray.shape
    is_clean = gray.mean()>235 and gray.std()<45          # already a clean digital drawing?
    if not is_clean:
        gray=_strip_frame(gray)
        if max(H,W)<500:                                  # upscale tiny images
            pil=Image.fromarray(gray); gray=np.array(pil.resize((pil.width*2,pil.height*2),Image.LANCZOS))
        k=max(15,(min(gray.shape)//12)|1)                 # flat-field: remove paper/stains/gradient
        bg=cv2.morphologyEx(gray,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k)))
        gray=cv2.normalize(cv2.divide(gray,bg,scale=255),None,0,255,cv2.NORM_MINMAX).astype(np.uint8)
        gray=cv2.bilateralFilter(gray,7,50,50)
    if not binarize: return gray
    _,b=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU); return b

In [ ]:
# 4) Recognise: multi-variant CONSENSUS + salt-stripping (the accuracy engine)
from collections import Counter
from rdkit.Chem.MolStandardize import rdMolStandardize
_chooser = rdMolStandardize.LargestFragmentChooser()
IMPROBABLE = set("Ga Ge As Se Sb Te Sn Pb Hg Tl Bi Al Si B Zn Fe Cu Ni Co Mn Cr".split())
_pubchem_cache = {}

def _variants(image_path, work_dir):
    os.makedirs(work_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(image_path))[0]
    v = {"original": image_path}
    bp = os.path.join(work_dir, base + "_bin.png");  cv2.imwrite(bp, clean_image(image_path, True))
    gp = os.path.join(work_dir, base + "_gray.png"); cv2.imwrite(gp, clean_image(image_path, False))
    v["cleaned_binary"] = bp; v["cleaned_gray"] = gp
    return v

def _decimer_model(path, hand_drawn=False):
    from DECIMER import predict_SMILES
    for kw in ({"confidence": True, "hand_drawn": hand_drawn}, {"confidence": True},
               {"hand_drawn": hand_drawn}, {}):
        try:
            out = predict_SMILES(path, **kw)
            if isinstance(out, tuple):
                try: conf = float(np.mean([c[1] for c in out[1]]))
                except Exception: conf = None
                return out[0], conf
            return out, None
        except TypeError:
            continue
    return None, None

def _molscribe_model(path, hand_drawn=False):
    try:
        global _MOLSCRIBE
        if "_MOLSCRIBE" not in globals():
            from molscribe import MolScribe
            import torch, huggingface_hub
            ckpt = huggingface_hub.hf_hub_download("yujieq/MolScribe", "swin_base_char_aux_1m.pth")
            _MOLSCRIBE = MolScribe(ckpt, device=torch.device("cpu"))
        r = _MOLSCRIBE.predict_image_file(path)
        return r["smiles"], r.get("confidence")
    except Exception:
        return None, None   # not installed -> skipped in the vote

MODELS = [("DECIMER", _decimer_model), ("MolScribe", _molscribe_model)]

def _sanitize(smiles):
    if not smiles: return None, {}
    m = Chem.MolFromSmiles(smiles)
    if m is None: return None, {}
    n_frag = len(Chem.GetMolFrags(m))
    m = _chooser.choose(m)
    elems = {a.GetSymbol() for a in m.GetAtoms()}
    return Chem.MolToSmiles(m), {
        "improbable": sorted(elems & IMPROBABLE),
        "halogens": sorted(elems & set("F Cl Br I".split())),
        "had_fragments": n_frag > 1,
    }

def _pubchem_ok(smiles):
    if not USE_PUBCHEM or not smiles: return None
    if smiles in _pubchem_cache: return _pubchem_cache[smiles]
    import time
    try:
        import pubchempy as pcp
        from rdkit.Chem import inchi as rdi
        ik = rdi.InchiToInchiKey(rdi.MolToInchi(Chem.MolFromSmiles(smiles)))
        ans = None
        for _ in range(3):
            try:
                ans = bool(pcp.get_compounds(ik, "inchikey")); break
            except Exception:
                time.sleep(1.0)
        _pubchem_cache[smiles] = ans
        return ans
    except Exception:
        return None

def recognize(image_path, work_dir):
    reads = []
    for vpath in _variants(image_path, work_dir).values():
        for mname, mfn in MODELS:
            for hd in ([False, True] if (TRY_HAND_DRAWN and mname == "DECIMER") else [False]):
                smi, conf = mfn(vpath, hand_drawn=hd)
                clean, info = _sanitize(smi)
                if clean and not info.get("improbable"):
                    reads.append((clean, conf or 0.0, bool(info.get("halogens")),
                                  bool(info.get("had_fragments"))))
    if not reads:
        return {"smiles": None, "trust": False, "votes": 0, "total": 0,
                "agreement": 0, "confidence": None, "needs_review": True}

    votes = Counter(r[0] for r in reads)
    best, n = votes.most_common(1)[0]
    total = len(reads)
    best_conf = max(r[1] for r in reads if r[0] == best)
    had_halogen = any(r[2] for r in reads if r[0] == best)
    agreement = n / total

    in_pc = _pubchem_ok(best)
    trust = (n >= 2 and agreement >= 0.5) or (best_conf >= 0.9 and len(votes) == 1)
    if in_pc is False and n < 2:
        trust = False
    needs_review = (not trust) or had_halogen

    return {"smiles": best, "trust": bool(trust), "votes": n, "total": total,
            "agreement": round(agreement, 2), "confidence": round(best_conf, 3),
            "in_pubchem": in_pc, "needs_review": bool(needs_review)}

In [ ]:
def redraw(smiles, out_base, width=1600, height=1200, bond_width=4):
    mol=Chem.MolFromSmiles(smiles)
    if mol is None: return None
    d=rdMolDraw2D.MolDraw2DCairo(width,height); d.drawOptions().bondLineWidth=bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d,mol); d.FinishDrawing()
    png=out_base+"_highres.png"; open(png,"wb").write(d.GetDrawingText())
    d2=rdMolDraw2D.MolDraw2DSVG(width,height); d2.drawOptions().bondLineWidth=bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d2,mol); d2.FinishDrawing()
    open(out_base+"_highres.svg","w").write(d2.GetDrawingText())
    return png

In [ ]:
# 6) Run on every image -> CSV + accuracy summary  (always redraws; trust is a LABEL)
work_dir = os.path.join(OUT_DIR, "_tmp")
rows = []; n = correct = trusted = 0

image_files = sorted(f for f in os.listdir(IMAGE_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))
                     and "_highres" not in f and "_cleaned" not in f)

for fname in image_files:
    path = os.path.join(IMAGE_DIR, fname); name = os.path.splitext(fname)[0]; n += 1
    try:
        best = recognize(path, work_dir)
    except Exception as e:
        rows.append({"image": fname, "smiles": f"ERROR: {e}", "trust": False,
                     "needs_review": True}); continue

    # ALWAYS draw the best structure so you can see every result; trust just tells you whether to double-check
    highres = redraw(best["smiles"], os.path.join(OUT_DIR, name)) if best["smiles"] else None

    correct_flag = None
    if name in GROUND_TRUTH:
        gt = Chem.MolFromSmiles(GROUND_TRUTH[name])
        gt = Chem.MolToSmiles(gt) if gt else None
        correct_flag = int(best["smiles"] == gt); correct += correct_flag
    trusted += int(best["trust"])

    rows.append({"image": fname, "smiles": best["smiles"], "votes": f"{best['votes']}/{best['total']}",
                 "agreement": best["agreement"], "confidence": best["confidence"],
                 "in_pubchem": best.get("in_pubchem"), "trust": best["trust"],
                 "needs_review": best["needs_review"], "highres": highres, "correct": correct_flag})
    tag = "OK   " if best["trust"] else "CHECK"
    print(f"[{tag}] {fname:28s} {best['votes']}/{best['total']} agree  {best['smiles']}")

df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)
print("\n================ SUMMARY ================")
print(f"images processed : {n}")
print(f"trusted          : {trusted}/{n}  (others are produced too, just flagged needs_review)")
n_gt = sum(1 for r in rows if r.get('correct') is not None)
if n_gt:
    print(f"EXACT-MATCH ACC  : {correct}/{n_gt} = {100*correct/n_gt:.0f}%   (vs GROUND_TRUTH)")
else:
    print("EXACT-MATCH ACC  : fill GROUND_TRUTH in cell 2 to measure real accuracy")
print(f"\nResults CSV: {os.path.join(OUT_DIR, 'results.csv')}")
df

/root/.data/DECIMER-V2/models.zip
... done downloading trained model!


/root/.data/DECIMER-V2/DECIMER_HandDrawn_model.zip
... done downloading trained model!


[15:46:19] Running LargestFragmentChooser
[15:46:27] Running LargestFragmentChooser
[15:46:30] Running LargestFragmentChooser
[15:46:33] Running LargestFragmentChooser
[15:46:36] Running LargestFragmentChooser
[15:46:39] Running LargestFragmentChooser


[OK   ] blurry1_aspirin.png          3/6 agree  CC(C)C[C@@H]1CCCCC1C(C)C


[15:46:43] Running LargestFragmentChooser
[15:46:46] Running LargestFragmentChooser
[15:46:49] Running LargestFragmentChooser
[15:46:52] Running LargestFragmentChooser
[15:46:55] Running LargestFragmentChooser
[15:46:58] Running LargestFragmentChooser


...

[15:52:54] Running LargestFragmentChooser
[15:52:57] Running LargestFragmentChooser
[15:53:00] Running LargestFragmentChooser
[15:53:00] Fragment: CC(=O)OCc1ccccc1C(=O)O
[15:53:00] New largest fragment: CC(=O)OCc1ccccc1C(=O)O (24)
[15:53:00] Fragment: [I-]
[15:53:03] Running LargestFragmentChooser
[15:53:09] Running LargestFragmentChooser
[15:53:09] Fragment: CC(=O)OCc1cccc(I)c1C(=O)O
[15:53:09] New largest fragment: CC(=O)OCc1cccc(I)c1C(=O)O (24)
[15:53:09] Fragment: [I-]
[15:53:09] Fragment: [I-]
[15:53:09] Fragment: [I-]
[15:53:09] Fragment: [I-]
[15:53:12] Running LargestFragmentChooser


[OK   ] molecule4.png                4/6 agree  CC(=O)OCc1cccc(I)c1C(=O)O


[15:53:17] Running LargestFragmentChooser
[15:53:22] Running LargestFragmentChooser
[15:53:25] Running LargestFragmentChooser
[15:53:29] Running LargestFragmentChooser
[15:53:33] Running LargestFragmentChooser
[15:53:46] Running LargestFragmentChooser


[CHECK] molecule5.png                1/6 agree  OCC1O[C@@H](O)C(O)[C@@H](O)[C@@H]1O

================ SUMMARY ================
images processed : 10
trusted          : 8/10  (others are produced too, just flagged needs_review)
EXACT-MATCH ACC  : fill GROUND_TRUTH in cell 2 to measure real accuracy

Results CSV: /content/output/results.csv


,image,smiles,votes,agreement,confidence,in_pubchem,trust,needs_review,highres,correct
0,blurry1_aspirin.png,CC(C)C[C@@H]1CCCCC1C(C)C,3/6,0.50,0.909,True,True,False,/content/output/blurry1_aspirin_highres.png,None
1,mix1_aspirin_lowres.png,CC(=O)Oc1ccccc1C(=O)O,5/6,0.83,0.955,True,True,False,/content/output/mix1_aspirin_lowres_highres.png,None
2,mix2_paracetamol_blur.png,CC(=O)Nc1ccc(O)cc1,4/6,0.67,0.967,True,True,False,/content/output/mix2_paracetamol_blur_highres.png,None
3,mix3_ibuprofen_paper.png,CC(C)Cc1ccc(C(C)C(=O)O)cc1,6/6,1.00,0.992,True,True,False,/content/output/mix3_ibuprofen_paper_highres.png,None
4,mix4_caffeine_scan.png,Cn1c(=O)c2c(ncn2C)n(C)c1=O,6/6,1.00,0.995,True,True,False,/content/output/mix4_caffeine_scan_highres.png,None
5,molecule.png,Cn1c(=O)c2c(ncn2C)n(C)c1=O,6/6,1.00,0.996,True,True,False,/content/output/molecule_highres.png,None
6,molecule1.png,Cn1c(=O)c2c(nc(CC34CC5CC(CC(C5)C3)C4)n2C)n(C)c1=O,1/3,0.33,0.821,False,False,True,/content/output/molecule1_highres.png,None
7,molecule2.png,c1ccccc1,4/6,0.67,0.856,True,True,False,/content/output/molecule2_highres.png,None
8,molecule4.png,CC(=O)OCc1cccc(I)c1C(=O)O,4/6,0.67,0.951,False,True,True,/content/output/molecule4_highres.png,None
9,molecule5.png,OCC1O[C@@H](O)C(O)[C@@H](O)[C@@H]1O,1/6,0.17,0.881,True,False,True,/content/output/molecule5_highres.png,None
